In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Mon Sep 15 04:29:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!gdown --id 13wBO3baamOfaJnFBK6xyFxsC7AD0eT-F
!gdown --id 13sU9VWS_WISO_e8wU8S7zTo04qNpixrc
!gdown --id 13sg-tQrqtMJ1vmU3DhmaYmtg792eIS1Q

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=13wBO3baamOfaJnFBK6xyFxsC7AD0eT-F
To: /content/prachatai-train.csv
100% 7.86M/7.86M [00:00<00:00, 48.3MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=13sU9VWS_WISO_e8wU8S7zTo04qNpixrc
To: /content/prachatai-dev.csv
100% 990k/990k [00:00<00:00, 12.8MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: 

In [3]:
import pandas as pd
train_df = pd.read_csv('prachatai-train.csv')
dev_df = pd.read_csv('prachatai-dev.csv')
test_df = pd.read_csv('prachatai-test.csv')

train_df['title tokens'] = train_df['title tokens'].apply(lambda x: x.replace('|', ''))
dev_df['title tokens'] = dev_df['title tokens'].apply(lambda x: x.replace('|', ''))
test_df['title tokens'] = test_df['title tokens'].apply(lambda x: x.replace('|', ''))

labels = list(set(train_df['label'].to_list()))
label_to_idx = {label:i for i,label in enumerate(labels)}
train_df['label'] = train_df['label'].apply(lambda x: label_to_idx[x])
dev_df['label'] = dev_df['label'].apply(lambda x: label_to_idx[x])
test_df['label'] = test_df['label'].apply(lambda x: label_to_idx[x])

label_list = [0] * len(label_to_idx)
for label in label_to_idx:
  label_list[label_to_idx[label]] = label

In [4]:
label_list

['สังคม', 'สิ่งแวดล้อม', 'เศรษฐกิจ', 'ต่างประเทศ', 'การเมือง', 'คุณภาพชีวิต']

In [6]:
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoConfig,

    Trainer,
    TrainingArguments,
)

from datasets import (
    Dataset,
    DatasetDict,
    Features, Sequence, ClassLabel, Value
)

In [7]:
#model_name = 'airesearch/wangchanberta-base-att-spm-uncased'
model_name = 'clicknext/phayathaibert'
config = AutoConfig.from_pretrained(model_name, num_labels=len(labels))
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                           config=config)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.4M [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at clicknext/phayathaibert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
features= Features({
                    "title tokens": Value(dtype='string'),
                    "label": ClassLabel(names=label_list)
                })
d = DatasetDict({'train': Dataset.from_pandas(train_df, features=features),
                 'dev': Dataset.from_pandas(dev_df, features=features),
                 'test': Dataset.from_pandas(test_df, features=features)})

In [9]:
d

DatasetDict({
    train: Dataset({
        features: ['title tokens', 'label'],
        num_rows: 37890
    })
    dev: Dataset({
        features: ['title tokens', 'label'],
        num_rows: 4736
    })
    test: Dataset({
        features: ['title tokens', 'label'],
        num_rows: 4737
    })
})

In [10]:
d['train'][0]

{'title tokens': 'กมธ.แจงมติเซ็ตซีโร่ กกต.หวั่นเป็นปลา 2 น้ำทำงานลำบาก ย้ำไม่มีใบสั่งจาก คสช.',
 'label': 4}

In [13]:
def tokenize(examples):
    tokenized_inputs = tokenizer(examples["title tokens"],
                                 is_split_into_words=False,
                                 truncation=True,
                                 max_length=50)
    return tokenized_inputs
tokenized_datasets = d.map(tokenize, batched=True)


Map:   0%|          | 0/37890 [00:00<?, ? examples/s]

Map:   0%|          | 0/4736 [00:00<?, ? examples/s]

Map:   0%|          | 0/4737 [00:00<?, ? examples/s]

In [14]:
tokenizer.convert_ids_to_tokens(tokenized_datasets['train'][0]['input_ids'])

['<s>',
 '▁',
 'กมธ',
 '.',
 'แจง',
 'มติ',
 'เซ็ต',
 'ซีโร่',
 '▁กกต',
 '.',
 'หวั่น',
 'เป็น',
 'ปลา',
 '▁2',
 '▁',
 'น้ํา',
 'ทํางาน',
 'ลําบาก',
 '▁',
 'ย้ํา',
 'ไม่มี',
 'ใบสั่ง',
 'จาก',
 '▁',
 'คสช',
 '.',
 '</s>']

In [15]:
model_name.split('/')[-1]

'phayathaibert'

In [16]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['title tokens', 'label', 'input_ids', 'attention_mask'],
        num_rows: 37890
    })
    dev: Dataset({
        features: ['title tokens', 'label', 'input_ids', 'attention_mask'],
        num_rows: 4736
    })
    test: Dataset({
        features: ['title tokens', 'label', 'input_ids', 'attention_mask'],
        num_rows: 4737
    })
})

In [17]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k != 'title tokens'}
[len(x) for x in samples["input_ids"]]

[27, 15, 25, 8, 11, 30, 23, 29]

In [19]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [23]:
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoConfig,

    Trainer,
    TrainingArguments,
)

from datasets import (
    Dataset,
    DatasetDict,
    Features, Sequence, ClassLabel, Value
)

args = TrainingArguments(
    output_dir='prachatai-headline'
)

args2 = TrainingArguments(
    output_dir='prachatai-headline-2',
    learning_rate=1e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    eval_strategy='steps',
    eval_steps=100,
    logging_steps=100,
    save_strategy='steps',
    report_to='none'
)

# pip install evaluate  (or use scikit-learn if you prefer)

import numpy as np
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # If model returns tuple (e.g., with past_key_values), take the first element
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)

    acc = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1w = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    # You can add "macro" too if you want:
    f1m = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]

    return {"accuracy": acc, "f1w": f1w, "f1m": f1m}


trainer = Trainer(
    model,
    args2,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["dev"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1w,F1m
100,0.946900,0.754017,0.747466,0.686901,0.307262
200,0.739600,0.659448,0.770693,0.739885,0.413933
300,0.644200,0.627279,0.783995,0.756671,0.458874
400,0.625500,0.597806,0.788851,0.769249,0.497247
500,0.611500,0.589634,0.790118,0.773147,0.519155
600,0.578900,0.574515,0.797297,0.779719,0.529206
700,0.570500,0.564743,0.797931,0.785564,0.551800
800,0.546900,0.568941,0.796030,0.787592,0.564237
900,0.531400,0.561287,0.801520,0.790314,0.559470
1000,0.545100,0.553207,0.803421,0.790896,0.558392


TrainOutput(global_step=1186, training_loss=0.6193441484267956, metrics={'train_runtime': 961.5039, 'train_samples_per_second': 78.814, 'train_steps_per_second': 1.233, 'total_flos': 1298484274128840.0, 'train_loss': 0.6193441484267956, 'epoch': 2.0})

In [24]:

predictions = trainer.predict(tokenized_datasets['test'])
preds = np.argmax(predictions.predictions, axis=-1)


In [25]:
predictions.label_ids

array([4, 1, 4, ..., 4, 1, 4])

In [26]:
preds

array([4, 4, 4, ..., 4, 4, 3])

In [27]:
predicted_labels = [labels[x] for x in preds]
gold = [labels[x] for x in predictions.label_ids]

In [28]:
from sklearn.metrics import classification_report
print(classification_report(gold, predicted_labels))

              precision    recall  f1-score   support

    การเมือง       0.86      0.91      0.88      3289
 คุณภาพชีวิต       0.60      0.40      0.48       290
  ต่างประเทศ       0.72      0.82      0.77       475
       สังคม       0.56      0.13      0.21       139
 สิ่งแวดล้อม       0.65      0.63      0.64       427
    เศรษฐกิจ       0.63      0.39      0.48       117

    accuracy                           0.81      4737
   macro avg       0.67      0.55      0.58      4737
weighted avg       0.79      0.81      0.79      4737



In [29]:
input = tokenizer("นักวิชาการเสนอใช้ 'สวัสดิการชราภาพถ้วนหน้าแบบยืดหยุ่น' ค้านพิสูจน์ความจนก่อนรับสิทธิ", return_tensors="pt")

In [30]:
input.to('cuda')
outputs = trainer.model(**input)
logits = outputs.logits
predicted_class = logits.argmax().item()

In [31]:
labels[predicted_class]

'คุณภาพชีวิต'